# NeuroVision v3 — Baseline-normalized SANS relationship explorer
Run install cell, then app cell. Built-in data are synthetic. CFD/FEA remain simplified teaching surrogates.

In [1]:
!pip -q install gradio pandas plotly


In [3]:
from datetime import datetime, timezone
import math
import gradio as gr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Fictional data solely to demonstrate the analysis workflow.
# Each series is aligned to the same illustrative mission-day schedule.
RAW = pd.DataFrame({
    "Mission day": [0, 15, 30, 45, 60, 75, 90],
    "Venous-flow proxy (a.u.)": [1.00, .96, .91, .88, .86, .90, .92],
    "CSF-dynamics proxy (a.u.)": [1.00, 1.03, 1.06, 1.08, 1.09, 1.07, 1.05],
    "Peripapillary thickness (µm)": [100, 101, 103, 105, 107, 106, 104],
    "Visual-function score (a.u.)": [100, 100, 99, 98, 97, 98, 99]
})
MEASURES = list(RAW.columns[1:])
def stamp(): return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
def blank(): return {"daily": [], "research": []}

def normalized_data():
    out = RAW[["Mission day"]].copy()
    for col in MEASURES:
        baseline = RAW[col].iloc[0]
        out[col + " · baseline index"] = RAW[col] / baseline * 100 if baseline else float("nan")
    return out

NORM = normalized_data()
NORM_COLS = list(NORM.columns[1:])

def daily_save(s,b,d,f,v,n):
    s=dict(s or blank())
    s["daily"]=(s["daily"]+[{"Recorded (UTC)":stamp(),"Blurred vision":b,"Distorted vision":d,
      "Focusing difficulty":f,"Visual-field change":v,"Notes":n or ""}])[-100:]
    msg=("**Change recorded.** This tool cannot determine the cause. Follow mission medical procedures and notify the flight surgeon/medical team promptly for new, worsening, sudden or persistent visual symptoms."
         if "Yes" in [b,d,f,v] else "Check-in recorded. This is not medical clearance and does not rule out SANS.")
    return s,msg,pd.DataFrame(s["daily"])

def research_save(s,a,dom,m,val,u,side,protocol,note):
    s=dict(s or blank()); s["research"]=(s["research"]+[{"Recorded (UTC)":stamp(),"Assessment":a,"Domain":dom,
      "Measurement":m,"Value":val,"Unit":u,"Eye/side":side,"Protocol":protocol,"Notes":note}])[-200:]
    return s,"Logged. Verify units, device, protocol and clinical source before interpretation.",pd.DataFrame(s["research"])

def timeline_raw():
    # Separate panels preserve original physical units; no mixed-unit y-axis.
    fig=go.Figure()
    for col in MEASURES:
        fig.add_trace(go.Scatter(x=RAW["Mission day"],y=RAW[col],mode="lines+markers",name=col))
    fig.update_layout(title="Raw measurements — separate units shown in hover/legend",
                      xaxis_title="Mission day (shared time axis)",yaxis_title="Raw value (units differ)",
                      height=470,legend=dict(orientation="h",y=-.28))
    # Four independent y-axes would still confuse; use faceted small multiples instead.
    long=RAW.melt(id_vars="Mission day",var_name="Measurement",value_name="Raw value")
    fac=px.line(long,x="Mission day",y="Raw value",facet_row="Measurement",markers=True,
                title="Raw measurements in separate panels (preserves each unit)")
    fac.update_layout(height=780,showlegend=False)
    fac.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    return fac

def timeline_normalized():
    long=NORM.melt(id_vars="Mission day",var_name="Measurement",value_name="Baseline index")
    fig=px.line(long,x="Mission day",y="Baseline index",color="Measurement",markers=True,
                title="Baseline-normalized trends — each series starts at 100")
    fig.add_hline(y=100,line_dash="dash",annotation_text="Personal baseline = 100")
    fig.update_layout(height=470,xaxis_title="Mission day (shared time axis)",
                      yaxis_title="Index (% of baseline)",legend=dict(orientation="h",y=-.3))
    return fig

def normalized_table():
    return NORM.round(2)

def correlation_matrix():
    # Correlations calculated from normalized series; linear rescaling leaves Pearson r unchanged.
    corr=NORM[NORM_COLS].corr()
    fig=px.imshow(corr,text_auto=".2f",zmin=-1,zmax=1,aspect="auto",
                  title="Pairwise correlation — synthetic normalized series")
    fig.update_layout(height=480)
    return fig

def relationship(x,y):
    df=NORM[["Mission day",x,y]].dropna()
    r=df[x].corr(df[y])
    fig=px.scatter(df,x=x,y=y,color="Mission day",hover_data=["Mission day"],
                   title="Normalized association across aligned mission days")
    fig.add_hline(y=100,line_dash="dash")
    fig.add_vline(x=100,line_dash="dash")
    note=(f"Pearson correlation: **{r:.2f}** across {len(df)} synthetic time points. "
          "Both axes are baseline indices (100 = each measure's own baseline). "
          "This is descriptive association only—not causation, a clinical threshold, or a validated prediction.")
    return fig,note

def lag_explorer(x,y,lag):
    # Positive lag means candidate mechanism at day t is compared with outcome at later day t+lag.
    a=NORM[["Mission day",x]].copy()
    b=NORM[["Mission day",y]].copy()
    b["Mission day"]=b["Mission day"]-lag
    joined=a.merge(b,on="Mission day",suffixes=("_mechanism","_outcome")).dropna()
    if len(joined)<3:
        return f"Only {len(joined)} aligned pairs at this lag; too few for even an exploratory correlation."
    r=joined[x].corr(joined[y])
    return (f"At an illustrative lag of **{lag} mission days**, {len(joined)} paired points give r = **{r:.2f}**. "
            "This is an exploratory timing comparison in synthetic data. It does not establish biological delay or causation.")

def cfd(radius_mm,length_mm,dp_pa,mu_mpas):
    r=radius_mm/1000; L=length_mm/1000; mu=mu_mpas/1000
    q=math.pi*r**4*dp_pa/(8*mu*L); vel=q/(math.pi*r*r); shear=4*mu*q/(math.pi*r**3)
    return (f"**Idealized flow rate:** {q*1e9:.4g} µL/s  \n**Mean velocity:** {vel*1000:.4g} mm/s  \n"
            f"**Wall shear estimate:** {shear:.4g} Pa  \n\n"
            "Steady laminar Newtonian flow in a rigid circular tube—not anatomical CFD of CSF around the optic nerve. It omits annular anatomy, pulsatility, compliant walls, tissue exchange and validated boundary conditions.")

def fea(stress_kpa,E_mpa,L_mm):
    strain=(stress_kpa*1000)/(E_mpa*1e6); disp=strain*(L_mm/1000)
    return (f"**Nominal strain:** {strain:.4g} ({strain*100:.4g}%)  \n**Displacement proxy:** {disp*1e6:.4g} µm  \n\n"
            "One-dimensional homogeneous linear-elastic proxy, not an FEA mesh or optic nerve tissue model.")

with gr.Blocks(title="NeuroVision | SANS Evidence Fusion") as app:
    state=gr.State(blank())
    gr.Markdown("""# NEUROVISION
### SANS Evidence Fusion · Baseline Normalization · CFD/FEA Research Concept

**Relationship analysis:** align measurements on a common mission timeline, preserve original units in separate panels, and compare relative changes using each measure's own baseline.

**Disclosure:** built-in records are synthetic. Manual entries are not connected to NASA systems or medical devices. This is not a diagnostic or validated prediction tool.""")
    with gr.Tabs():
        with gr.Tab("1 · Daily check-in"):
            gr.Markdown("Record SANS-relevant visual symptoms. Report new or worsening symptoms under mission medical procedures.")
            with gr.Row():
                b=gr.Radio(["No","Yes","Unsure"],label="Blurred/reduced visual clarity?")
                d=gr.Radio(["No","Yes","Unsure"],label="Distorted/wavy vision?")
            with gr.Row():
                f=gr.Radio(["No","Yes","Unsure"],label="New focusing difficulty?")
                v=gr.Radio(["No","Yes","Unsure"],label="New visual-field change/blind spot?")
            n=gr.Textbox(label="Optional notes")
            btn=gr.Button("Save daily check-in",variant="primary"); msg=gr.Markdown(); daily=gr.Dataframe(label="Observation history")
            btn.click(daily_save,[state,b,d,f,v,n],[state,msg,daily])
        with gr.Tab("2 · Periodic measurements"):
            gr.Markdown("Log clinician-supervised or research measurements with units, eye/side, device and protocol.")
            a=gr.Dropdown(["Pre-flight baseline","Weekly research assessment","Monthly research assessment","Clinically indicated assessment","Post-flight follow-up"],value="Monthly research assessment",label="Assessment")
            dom=gr.Dropdown(["Ocular structure","Visual function","Fluid / vascular mechanism","Brain / neuro-ocular research"],value="Ocular structure",label="Domain")
            m=gr.Dropdown(["OCT — peripapillary retinal thickness / RNFL","OCT — optic disc edema","OCT — ganglion cell layer","OCT / fundoscopy — chorioretinal folds","MRI — posterior globe shape","Refraction","Visual acuity","Visual field","Doppler — internal jugular venous flow","CSF / intracranial fluid dynamics research","MRI — brain / fluid distribution","Other"],label="Measurement")
            with gr.Row():
                val=gr.Textbox(label="Value / finding"); u=gr.Textbox(label="Unit")
                side=gr.Dropdown(["Both eyes","Right eye","Left eye","Not applicable / unspecified"],value="Both eyes",label="Eye / side")
            protocol=gr.Textbox(label="Instrument / protocol"); note=gr.Textbox(label="Notes / source")
            rb=gr.Button("Log measurement",variant="primary"); rm=gr.Markdown(); rdf=gr.Dataframe(label="Research measurement history")
            rb.click(research_save,[state,a,dom,m,val,u,side,protocol,note],[state,rm,rdf])
        with gr.Tab("3 · Relationship explorer"):
            gr.Markdown("""### A. Raw measurements, same mission clock
Each variable is displayed in its own panel so µm, arbitrary scores and flow proxies are never treated as interchangeable.""")
            gr.Plot(value=timeline_raw())
            gr.Markdown("""### B. Baseline-normalized comparison
Each variable is divided by its own day-0 value and multiplied by 100. A value of 105 means 5% above that measure's own baseline—not the same physical quantity as another variable.""")
            gr.Plot(value=timeline_normalized())
            gr.Dataframe(value=normalized_table(),label="Normalized values (% of each measure's own baseline)")
            gr.Markdown("### C. Pairwise relationships")
            gr.Plot(value=correlation_matrix())
            with gr.Row():
                x=gr.Dropdown(NORM_COLS,value=NORM_COLS[1],label="Candidate mechanism (normalized)")
                y=gr.Dropdown(NORM_COLS,value=NORM_COLS[2],label="Ocular / functional outcome (normalized)")
            ex=gr.Button("Explore normalized relationship",variant="primary"); p=gr.Plot(); t=gr.Markdown()
            ex.click(relationship,[x,y],[p,t])
            gr.Markdown("### D. Exploratory time-lag comparison")
            lag=gr.Slider(0,30,value=15,step=15,label="Candidate mechanism lead time (mission days)")
            lagbtn=gr.Button("Explore lagged association"); lagout=gr.Markdown()
            lagbtn.click(lag_explorer,[x,y,lag],lagout)
            gr.Markdown("""**Interpretation safeguards:** the example uses seven synthetic time points at 15-day intervals. Normalization improves relative visualization but does not make measurements biologically equivalent. Correlation and lagged association do not establish causation. Real analysis needs validated measurements, individual baselines, consistent protocols, adequate sample sizes, confounder handling, uncertainty estimates and missing-data methods.""")
        with gr.Tab("4 · CFD / FEA workbench"):
            gr.Markdown("Transparent mathematical teaching surrogates only—not full anatomical CFD or FEA solvers.")
            gr.Markdown("### CFD · rigid circular-tube flow sensitivity")
            with gr.Row():
                rad=gr.Slider(.2,3,value=1,step=.1,label="Tube radius (mm)")
                leng=gr.Slider(5,50,value=20,step=1,label="Tube length (mm)")
            with gr.Row():
                dp=gr.Slider(0,100,value=5,step=1,label="Pressure difference (Pa)")
                mu=gr.Slider(.5,5,value=1,step=.1,label="Dynamic viscosity (mPa·s)")
            cb=gr.Button("Calculate idealized flow",variant="primary"); co=gr.Markdown(); cb.click(cfd,[rad,leng,dp,mu],co)
            gr.Markdown("### FEA · one-dimensional elastic response proxy")
            with gr.Row():
                sig=gr.Slider(0,20,value=2,step=.5,label="Nominal stress (kPa)")
                young=gr.Slider(.05,5,value=1,step=.05,label="Young's modulus (MPa)")
                char=gr.Slider(.1,10,value=2,step=.1,label="Characteristic length (mm)")
            fb=gr.Button("Calculate elastic proxy",variant="primary"); fo=gr.Markdown(); fb.click(fea,[sig,young,char],fo)
            gr.Markdown("A genuine coupled model requires anatomical geometry, fluid and tissue properties, pulsatile/venous boundary conditions, constitutive laws, mesh and solver verification, sensitivity analysis and validation against ocular measurements.")
    gr.Markdown("--- **Scientific scope:** candidate mechanisms include headward fluid shifts, venous/vascular changes and CSF dynamics; no single cause is assumed. Brain-related links remain exploratory. All built-in data are synthetic.")
if __name__=="__main__":
    app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://299c3ad45cb6f8cde9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
